A notebook to load in the old escape tuning curves and make some heatmaps sorted by them  
- manual labels
- escapes
- auto labels
- rejected auto labels

In [13]:
%load_ext autoreload
%autoreload 2

from databank import full_experiments_objects
from behave_analysis.process.process import Process
from settings.settings_overrides import settings_overrides
from behave_analysis.analyze.results_database_utils import check_database_for_same_run, settings_to_check
from settings.settings_analyze_efizz import Settings_ae
from behave_analysis.analyze.EscapePattern.escape_pattern_utils import compute_tuning_stat
from behave_analysis.analyze.behaviour.homings_escapes.homing_load_manual_labels import load_manual_labels
from behave_analysis.analyze.behaviour.homings_escapes.homings import get_Homings
from behave_analysis.utils.arena_plotting import Arena
from behave_analysis.utils.creating_directories import make_directory
from behave_analysis.utils.identify_condition import build_condition_bool, build_flippedbarrier_condition_bool

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import polars as pl
import os

%matplotlib inline
time_period = 'to_subgoal_homing&escape' # 'homing&escape', 'correct_full_homing&escape' and 'to_subgoal_homing&escape'
var = 'frac_route in ' + time_period  
qualifier = "_bc"
save_folder = make_directory(f"Z:\Jasmine_Laurence\summary_plots\Sequence_auto_{time_period}{qualifier}")


tuning_settings = {'ep_bins': 25,
                    'ep_no_stationary': False,
                    'ep_interpolation_mult': 2,
                    'ep_gaussian_fitting': False,
                    'ep_compute_loo_reliability': False,
                    'ep_tuned_compare_method': 'euclidean',
                    'ep_tuned_stats': 'bootstrap',
                    'ep_tuned_stats_samples': 100,
                    'linshift_min_step': 120,
                    'linshift_step': 80,
                    'linshift_step_n': 100,
                    'stim_type': 'audio',
                    'cluster_type': 'good',
                    'cluster_labels': "bombcell",
                    "homings": 'auto',
                    'condition_types': 'experimental_conditions',
                    'compartment_split': ['all'],
                    'redo_compute': False}
Settings_ae = settings_overrides(Settings_ae, tuning_settings)

h_settings = {"homings_speed_threshold": 4.0,  # cm/s, used to find bouts of running that may be homings
            "homings_gap_tolerance": 1,  # frames, used to merge bouts
            "homings_features_initial_window_s": 1.0,  # seconds, used to compute initial features of homings like acceleration and hdir change
            "homing_classification_target_recall": 0.9,  # minimum recall for a gate to be considered valid
            "homings_classification_recall_threshold": 0.9,  # minimum recall for a feature gate to be considered valid
            "homings_classification_precision_threshold": 0.1,  # minimum precision for a feature gate to be considered valid
            "homings_classification_auc_threshold": 0.9,  # or .8, minimum AUC for a feature gate to be considered valid
            "homings_classification_cohens_d_threshold": 1,  # minimum absolute Cohen's d for a feature gate to be considered valid
            "redo_compute": False,
            "homings_use_boris": False,
            "homings_curated": False,
            "homings_distance_threshold": 25  # in cm, minimum length to be kept as a homings
            }
from settings.settings_analyze_behave import settings_ab
settings_ab = settings_overrides(settings_ab, h_settings)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
"""Run heatmap maker for all listed sessions"""
for exp in full_experiments_objects:
    session = Process(exp).load_session()
    print(f"Session name: {session['mouse']}_{session['experiment']}_{session['date']}")

    results_dict = {}
    settings_dict = {"variable": var, 
                    "insufficient_data": False, 
                    **settings_to_check(Settings_ae, ["ep_"])}
    savepath = os.path.join(session["base_path"], session["processed_path"], "models", "escape_tuning")
        
    database, do_analysis, hexaname = check_database_for_same_run(settings_dict, 
                                                                    savepath + os.sep + "EscapePattern_results.csv", 
                                                                    Settings_ae)
    if do_analysis:
        print("No matching results found in database.Try another session!.")
        continue

    # load data
    session_start = max(session["shelter_time"][0]*60*40, 30*40) # start at shelter_time or 30s, whichever is later
    video_df = pl.read_csv(os.path.join(session["base_path"], session["processed_path"]) + "\\" + "full_video_dataframe.csv")
    fcm = np.load(os.path.join(session["base_path"], session["processed_path"], "frame_by_good_cluster_matrix" + qualifier + ".npy"))
    mean_fcm = np.nanmean(fcm, axis=0)
    std_fcm = np.nanstd(fcm, axis=0)
    fcm_z = (fcm - mean_fcm) / std_fcm

    # load escape tuning
    data_file = os.path.join(savepath, f"EPtuning_{hexaname}_results.npz")
    ep_data = np.load(data_file, allow_pickle=True)
    real_stat, shift_stat = compute_tuning_stat(stat='zscore_peak', 
                                                shifted_matrix=ep_data['fr_shift'], 
                                                shift0=int(np.shape(ep_data['fr_shift'])[0] / 2), 
                                                neural_matrix=ep_data['neural_matrix'], 
                                                condition=ep_data['condition_vector'])
    sig_cells = real_stat > np.nanpercentile(shift_stat, 95, axis=0)
    cond_list = ep_data['all_conditions']

    # load manual homing onsets
    man_bool = np.zeros(len(video_df), dtype=bool)
    if os.path.isfile(session["base_path"] + '/' + session["processed_path"] + '/Borris/scored_homings.csv'):
        man_on, _, man_off = load_manual_labels(session)
        for on, off in zip(man_on, man_off):
            man_bool[on:off] = True
    else:
        print("No manual homing labels found for this session.")

    # load escape onsets
    esc_bool = np.zeros(len(video_df), dtype=bool)
    esc_path = os.path.join(session["base_path"], session["processed_path"], "escapes", "escapes.npy")
    if os.path.exists(esc_path):
        escapes = np.load(esc_path, allow_pickle=True).item()
        esc_on, esc_off = escapes["onset_frames"], escapes["offset_frames"]
        for on, off in zip(esc_on, esc_off):
            if np.isnan(on) or np.isnan(off):
                continue
            esc_bool[on:off] = True

    # load auto homing onsets
    h_bool = np.zeros(len(video_df), dtype=bool)
    homings_dict = get_Homings(settings_ab, session).get_homings(video_df=[], tracking_data=[]) 
    keep = homings_dict["removed_runs"] == False if "removed_runs" in homings_dict else np.ones(len(homings_dict["onset_frames"]), dtype=bool)
    h_on = homings_dict["onset_frames"][keep]
    h_off = homings_dict["offset_frames"][keep]
    for on, off in zip(h_on, h_off):
        h_bool[on:off] = True

    # separate curation-removed homing onsets
    not_h_bool = np.zeros(len(video_df), dtype=bool)
    if "removed_runs" in homings_dict:
        keep = homings_dict["removed_runs"] == True if "removed_runs" in homings_dict else np.zeros(len(homings_dict["onset_frames"]), dtype=bool)
        not_h_on = homings_dict["onset_frames"][keep]
        not_h_off = homings_dict["offset_frames"][keep]
        for on, off in zip(not_h_on, not_h_off):
            not_h_bool[on:off] = True

    # make condition vector
    bar = build_condition_bool(time_list = session["barrier_time"], cond_name = 'barrier', frame_idx=np.arange(len(video_df)) + 1, n_frames=len(video_df), fps = session["video"]["fps"])
    barflip = build_flippedbarrier_condition_bool(flip_time=session["barrier_flip_time"], frame_idx=np.arange(len(video_df)) + 1, n_frames=len(video_df), fps=session["video"]["fps"])
    shelter = build_condition_bool(time_list = session["shelter_time"], cond_name = 'shelter', frame_idx=np.arange(len(video_df)) + 1, n_frames=len(video_df), fps = session["video"]["fps"])

    # experimental condition vector
    condition = np.zeros(len(bar)) # nothing there
    if 'pre_shelter' in ep_data["all_conditions"]:
        condition[shelter == True] += 1 # shelter present
    condition[bar == True] += 1 # barrier is present
    condition[(bar == True) & (barflip == True)] += 1 # barrier is present and flipped
    if (np.sum(bar == True) > 0) & (bar[-1] == False):
        bar_removed = np.where(np.diff(bar.astype(int)) < 0)[0][0] + 1
        condition[bar_removed:] = np.amax(condition)+1 # barrier was removed

    for c in range(len(np.unique(condition))):
        """Plot the heatmaps one under the other, widths proportional to frame count"""
        cell_cond = c
        cond = c
        cap = 40
        cap_offset = 0
        bool_names = ["manual homings", "escapes", "auto homings", "removed homings"]
        bool_vecs = [man_bool, esc_bool, h_bool, not_h_bool]

        # Significant cells for this condition
        sig_mask = sig_cells[cell_cond, :]
        n_sig = int(np.sum(sig_mask))
        has_sig_cells = n_sig > 0

        if has_sig_cells:
            tuning_hm = ep_data['fr_full'][cell_cond, sig_mask, :]
            tuning_hm = np.divide(
                tuning_hm - mean_fcm[sig_mask, np.newaxis],
                std_fcm[sig_mask, np.newaxis],
                out=np.zeros_like(tuning_hm, dtype=np.float64),
                where=std_fcm[sig_mask, np.newaxis] != 0
            )
            sort_idx = np.argsort(np.nanargmax(tuning_hm, axis=1))
        else:
            sort_idx = None

        # --- Pre-compute frame data for all behaviors to get proportional widths ---
        behavior_data = []
        for bool_name, bool_vec in zip(bool_names, bool_vecs):
            indices_this_cond = np.where(np.logical_and(condition == cond, bool_vec))[0]
            if len(indices_this_cond) == 0: # no homings for this condition
                behavior_data.append(None)
                continue
            starts = np.concatenate(([0], np.where(np.diff(indices_this_cond) > 1)[0] + 1))
            ends = np.concatenate((starts[1:], [len(indices_this_cond)]))
            if len(starts) > cap:
                if len(starts) > cap + cap_offset:
                    starts = starts[cap_offset:cap_offset + cap]
                    ends = ends[cap_offset:cap_offset + cap]
                else:
                    starts = starts[-cap:]
                    ends = ends[-cap:]
            chunks = [indices_this_cond[s:e] for s, e in zip(starts, ends)]
            durations = np.array([len(chunk) for chunk in chunks], dtype=int)
            indices_concat = np.concatenate(chunks) if len(chunks) > 0 else np.array([], dtype=int)
            behavior_data.append({
                'indices_concat': indices_concat,
                'chunks': chunks,
                'durations': durations,
                'total_frames': int(np.sum(durations)),
            })

        frame_counts = [d['total_frames'] for d in behavior_data if d is not None]
        if len(frame_counts) == 0:
            print(f"No frames found for any behavior in condition {cond_list[cond]}. Skipping.")
            continue
        max_frames = max(d['total_frames'] for d in behavior_data if d is not None)

        # --- Layout constants (figure-coordinate fractions) ---
        fig_left   = 0.08
        fig_right  = 0.95
        fig_top    = 0.97
        fig_bottom = 0.04

        full_w   = fig_right - fig_left          # width available for the largest heatmap
        pair_h   = (fig_top - fig_bottom) / 4   # height budget per behavior row
        arena_h  = pair_h * 0.28
        hm_h     = pair_h * 0.65
        gap_h    = pair_h * 0.07                 # space between arena bottom and heatmap top

        fig = plt.figure(figsize=(max_frames/100, 28))

        for i, (bool_name, bool_vec, bdata) in enumerate(zip(bool_names, bool_vecs, behavior_data)):
            if bdata is None:
                print(f"No frames found for {bool_name} in condition {cond_list[cond]}. Skipping.")
                continue

            indices_this_cond = bdata['indices_concat']
            chunks = bdata['chunks']
            durations = bdata['durations']
            total_w   = float(np.sum(durations))

            # Proportional heatmap width (shared x-axis: 1 pixel = same frames everywhere)
            prop_w = (total_w / max_frames) * full_w

            # Vertical positions (stacking from top downward)
            pair_bottom = fig_top - (i + 1) * pair_h
            hm_bottom    = pair_bottom
            arena_bottom = hm_bottom + hm_h + gap_h

            neural = fcm_z[indices_this_cond, :][:, sig_mask]
            ax_hm = fig.add_axes([fig_left, hm_bottom, prop_w, hm_h])

            if has_sig_cells:
                heatmap = neural[:, sort_idx].T
                boundary_x = np.cumsum(durations)[:-1]

                ax_hm.imshow(
                    heatmap,
                    aspect="auto",
                    origin="lower",
                    cmap="gray_r",
                    vmin=-0.5, vmax=2,
                    interpolation="none"
                )
                for x in boundary_x:
                    ax_hm.axvline(x=x, color="red", linestyle="--", linewidth=1.5)
            else:
                ax_hm.text(
                    0.5, 0.5, "no significant cells",
                    ha="center", va="center",
                    transform=ax_hm.transAxes, fontsize=12
                )
                ax_hm.set_xticks([])
                ax_hm.set_yticks([])

            ax_hm.set_xlabel(f"Concatenated {bool_name} time (frames)")
            ax_hm.set_ylabel("Cells (sorted)")

            # --- Arena placement above this heatmap ---
            fig_w, fig_h = fig.get_size_inches()
            interval_starts  = np.concatenate(([0], np.cumsum(durations[:-1])))
            interval_centers = interval_starts + durations / 2.0
            min_dur = np.min(durations)

            side_from_data_in   = (prop_w * (min_dur / total_w) * 0.9) * fig_w
            max_side_arena_in   = (arena_h * 0.90) * fig_h
            max_side_abs_in     = 1.6
            side_in = min(side_from_data_in, max_side_arena_in, max_side_abs_in)

            top_w = side_in / fig_w
            top_h = side_in / fig_h
            top_y = arena_bottom + (arena_h - top_h) / 2.0

            for frames, c_mid in zip(chunks, interval_centers):
                cx   = fig_left + (c_mid / total_w) * prop_w
                left = np.clip(cx - top_w / 2.0, fig_left, fig_left + prop_w - top_w)

                ax_xy = fig.add_axes([left, top_y, top_w, top_h])
                Arena(
                    ax=ax_xy,
                    condition=cond_list[cond] + ("_tiny" if "tiny" in session["experiment"] else ""),
                    barrier_coordinates=session["barrier_location"][:-1] if session["barrier_location"] is not None else None,
                    full_image=False
                )
                prog = np.linspace(0, 1, len(frames)) if len(frames) > 1 else np.array([0.0])
                ax_xy.scatter(
                    video_df["mouse_x_position"].to_numpy()[frames],
                    video_df["mouse_y_position"].to_numpy()[frames],
                    c=prog, cmap="cool", vmin=0, vmax=1,
                    s=3, alpha=0.35
                )
                ax_xy.set_xticks([])
                ax_xy.set_yticks([])
        filename = f"{session['mouse']}_{session['experiment']}_{session['date']}_cond{cond}_new.png"
        plt.savefig(os.path.join(save_folder, filename), dpi=300, bbox_inches='tight')
        plt.close()

Session name: JAL004_shelter_2023_08_17
No matching results found in database.Try another session!.
Session name: JAL004_mushroom_2023_08_18
No matching results found in database.Try another session!.
Session name: JAL004_flip_2023_08_21


2026-08-14 20:36:39.001 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:24 - Found 4 matched results in database: ['36ba79cf6bd143cb' 'f4bf5d4a78d84c36' '24d092a8fbec454c'
 '3cbc56014f1941f3'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL004\004_flip_2023_08_21T12_53_10\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-08-14 20:36:45.774 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homing_load_manual_labels:load_manual_labels:21 - Loaded manual labels
2026-08-14 20:36:45.774 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homing_load_manual_labels:load_manual_labels:22 - Number of homings: 12
2026-08-14 20:36:45.774 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings:__init__:43 - checking for existing homings results
2026-08-14 20:36:45.796 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:24 - Found 3 matched results in database:

No frames found for removed homings in condition shelter_only. Skipping.
No frames found for removed homings in condition barrier_pre_flip. Skipping.
No frames found for manual homings in condition barrier_post_flip. Skipping.
No frames found for removed homings in condition barrier_post_flip. Skipping.


2026-08-14 20:37:05.785 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:24 - Found 3 matched results in database: ['628cce47dfad4028' '783ee111f26a4611' '3dd50fbdafcb4ebb'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL004\004_mush1_2023_08_22T13_13_41\processed_data\models\escape_tuning\EscapePattern_results.csv


Session name: JAL004_mushroom_2023_08_22


...

No frames found for removed homings in condition shelter_only. Skipping.
No frames found for removed homings in condition barrier_pre_flip. Skipping.
No frames found for removed homings in condition barrier_post_flip. Skipping.


2026-08-14 20:54:05.698 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:24 - Found 2 matched results in database: ['344610dad9494f0a' 'd628c26969e1438c'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_tiny_flip_2_2024_05_21T11_10_19\processed_data\models\escape_tuning\EscapePattern_results.csv


Session name: JAL008_tiny_2024_05_21


2026-08-14 20:54:09.939 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homing_load_manual_labels:load_manual_labels:21 - Loaded manual labels
2026-08-14 20:54:09.940 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homing_load_manual_labels:load_manual_labels:22 - Number of homings: 63
2026-08-14 20:54:09.962 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings:__init__:43 - checking for existing homings results
2026-08-14 20:54:09.976 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:24 - Found 2 matched results in database: ['fc6c2f12588742a7' 'ab69041b33f246e4'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_tiny_flip_2_2024_05_21T11_10_19\processed_data\homings\Homing_database.csv
2026-08-14 20:54:09.978 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings:get_homings:60 - Homing analysis already done with these settings, loading from database...


No frames found for removed homings in condition shelter_only. Skipping.
No frames found for removed homings in condition barrier_pre_flip. Skipping.
No frames found for removed homings in condition barrier_post_flip. Skipping.
